In [4]:
import sqlite3
import pandas as pd


providers = ["google", "awsX86", "awsARM", "oracleX86", "oracleARM", "deno", "cloudflare", "flyio", "fastly"]
column = "http_req_waiting"
db_path = "./data_copy.db"

cnx = sqlite3.connect(db_path)


def get_provider_df(cnx, provider: str) -> pd.DataFrame:
    """Load and lightly clean provider data."""
    
    if provider == "fastly":
        query = """
        SELECT http_req_waiting, language, memory
        FROM data
        WHERE provider = ?
          AND experiment = 'warmstart'
        """
        params = [provider]
    else:
        query = """
        SELECT http_req_waiting, language, memory
        FROM data
        WHERE provider = ?
          AND experiment = 'warmstart'
          AND cold_starts = 0
        """
        params = [provider]

    df = pd.read_sql_query(query, cnx, params=params)

    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found for provider '{provider}'.")

    df[column] = pd.to_numeric(df[column], errors="coerce")
    df = df.dropna(subset=[column, "memory", "language"])

    return df


def get_provider_df_cold(cnx, provider: str) -> pd.DataFrame:
    """Load and lightly clean provider data for coldstart (cold_starts=1)."""
    query = """
    SELECT http_req_waiting, language, memory
    FROM data
    WHERE provider = ?
      AND experiment = 'coldstart'
      AND cold_starts = 1
    """
    params = [provider]

    df = pd.read_sql_query(query, cnx, params=params)

    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found for provider '{provider}'.")

    df[column] = pd.to_numeric(df[column], errors="coerce")
    df = df.dropna(subset=[column, "memory", "language"])

    return df


def summarize_series(grouped: pd.core.groupby.generic.SeriesGroupBy) -> pd.DataFrame:
    """Standard (traffic-weighted) summary over raw samples."""
    def tail_ratio(s: pd.Series) -> float:
        mean = s.mean()
        return float(s.quantile(0.99) / mean) if mean and pd.notna(mean) else float("nan")

    return grouped.agg(
        count="size",
        mean="mean",
        median="median",
        std="std",
        p25=lambda s: s.quantile(0.25),
        p75=lambda s: s.quantile(0.75),
        p95=lambda s: s.quantile(0.95),
        p99=lambda s: s.quantile(0.99),
        iqr=lambda s: s.quantile(0.75) - s.quantile(0.25),
        tail_ratio=tail_ratio,
    ).reset_index()


def summarize_per_category(df_in: pd.DataFrame) -> pd.DataFrame:
    """Summaries per (memory, language) category (the 6 configs)."""
    def tail_ratio(s: pd.Series) -> float:
        mean = s.mean()
        return float(s.quantile(0.99) / mean) if mean and pd.notna(mean) else float("nan")

    return (
        df_in.groupby(["memory", "language"])[column]
        .agg(
            count="size",
            mean="mean",
            median="median",
            std="std",
            p25=lambda s: s.quantile(0.25),
            p75=lambda s: s.quantile(0.75),
            p95=lambda s: s.quantile(0.95),
            p99=lambda s: s.quantile(0.99),
            iqr=lambda s: s.quantile(0.75) - s.quantile(0.25),
            tail_ratio=tail_ratio,
        )
        .reset_index()
    )


def provider_summary_table(df_provider: pd.DataFrame, provider: str) -> pd.DataFrame:
    """Build the result table for one provider, including fair median-of-medians."""
    if df_provider.empty:
        return pd.DataFrame(columns=["provider", "level", "memory", "language",
                                     "count", "mean", "median", "std", "p25", "p75", "p95", "p99", "iqr", "tail_ratio"] )

    
    combo_rows = summarize_per_category(df_provider)
    combo_rows["level"] = "memory+language"

    
    overall_raw = summarize_series(df_provider[column].groupby(lambda _: "overall_raw"))
    overall_raw["level"] = "overall_raw"
    overall_raw["memory"] = "all"
    overall_raw["language"] = "all"
    overall_raw = overall_raw.drop(columns=["level_0"], errors="ignore")

    
    overall_fair = pd.DataFrame([{
        "level": "overall_fair",
        "memory": "all",
        "language": "all",
        "count": int(combo_rows["count"].sum()),        
        "mean": float(combo_rows["mean"].mean()),       
        "median": float(combo_rows["median"].median()), 
        "std": float(combo_rows["std"].median()),       
        "p25": float(combo_rows["p25"].median()),
        "p75": float(combo_rows["p75"].median()),
        "p95": float(combo_rows["p95"].median()),
        "p99": float(combo_rows["p99"].median()),
        "iqr": float(combo_rows["iqr"].median()),
        "tail_ratio": float(combo_rows["tail_ratio"].median()),
    }])

    
    memory_rows = summarize_series(df_provider.groupby("memory")[column])
    memory_rows["level"] = "memory"
    memory_rows["language"] = "all"

    language_rows = summarize_series(df_provider.groupby("language")[column])
    language_rows["level"] = "language"
    language_rows["memory"] = "all"

   
    result = pd.concat(
        [overall_raw, overall_fair, memory_rows, language_rows, combo_rows],
        ignore_index=True,
    )

    
    result.insert(0, "provider", provider)

    
    result = result[["provider", "level", "memory", "language", "count",
                     "mean", "median", "std", "p25", "p75", "p95", "p99", "iqr", "tail_ratio"]]

    return result


all_results = []

for p in providers:
    print("\n" + "=" * 90)
    print(f"PROVIDER: {p}")
    print("=" * 90)

    df_p = get_provider_df(cnx, p)

    if df_p.empty:
        print("No data after filtering/cleaning.")
        continue

    table = provider_summary_table(df_p, p)

    df_p_cold = get_provider_df_cold(cnx, p)
    if not df_p_cold.empty:
        table_cold = provider_summary_table(df_p_cold, p)
        cold_medians = table_cold[["provider", "level", "memory", "language", "median"]].rename(
            columns={"median": "median_cold"}
        )
        table = table.merge(cold_medians, on=["provider", "level", "memory", "language"], how="left")
        table["coldstart_factor"] = table["median_cold"] / table["median"]
        table = table.drop(columns=["median_cold"], errors="ignore")
    else:
        table["coldstart_factor"] = float("nan")

    display_cols = ["level", "memory", "language", "count", "median", "mean", "std", "p25", "p75", "p95", "p99", "iqr", "tail_ratio", "coldstart_factor"]
    print(table[display_cols].to_string(index=False))

    all_results.append(table)


combined = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

combined


PROVIDER: google
          level memory language   count    median      mean       std       p25       p75        p95        p99      iqr  tail_ratio
    overall_raw    all      all 1919360 42.525534 45.904602 30.721298 41.266987 47.500697  55.075430 110.091100 6.233711    2.398258
   overall_fair    all      all 1919360 42.456086 45.904418 26.555182 41.306052 47.306006  52.307725  66.691828 5.791124    1.505308
         memory 1024MB      all  639603 42.481256 44.644648 37.411556 41.291281 47.097015  51.918538  66.700075 5.805734    1.494022
         memory  128MB      all  639918 42.738216 48.549527 22.307390 41.355141 48.438763  83.807667 133.694587 7.083622    2.753777
         memory  512MB      all  639839 42.385624 44.518838 30.395275 41.158554 46.763570  51.930892  66.284412 5.605016    1.488907
       language    all       go  959761 42.001409 44.757168 23.712773 40.862581 46.781298  52.490778  85.799859 5.918717    1.917008
       language    all       js  959599 42.967019 4

,provider,level,memory,language,count,mean,median,std,p25,p75,p95,p99,iqr,tail_ratio
0,google,overall_raw,all,all,1919360,45.904602,42.525534,30.721298,41.266987,47.500697,55.075430,110.091100,6.233711,2.398258
1,google,overall_fair,all,all,1919360,45.904418,42.456086,26.555182,41.306052,47.306006,52.307725,66.691828,5.791124,1.505308
2,google,memory,1024MB,all,639603,44.644648,42.481256,37.411556,41.291281,47.097015,51.918538,66.700075,5.805734,1.494022
3,google,memory,128MB,all,639918,48.549527,42.738216,22.307390,41.355141,48.438763,83.807667,133.694587,7.083622,2.753777
4,google,memory,512MB,all,639839,44.518838,42.385624,30.395275,41.158554,46.763570,51.930892,66.284412,5.605016,1.488907
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,fastly,memory,na,all,640156,11.246481,8.941088,92.782407,7.365020,10.862982,17.561679,45.142443,3.497962,4.013917
91,fastly,language,all,go,320083,12.026494,10.124105,22.878713,8.892096,11.895150,18.856653,44.809176,3.003054,3.725872
92,fastly,language,all,js,320073,10.466443,7.612048,129.200531,6.590464,9.010774,15.575883,45.517961,2.420310,4.348943
93,fastly,memory+language,na,go,320083,12.026494,10.124105,22.878713,8.892096,11.895150,18.856653,44.809176,3.003054,3.725872


In [6]:
# Coldstart stats (exclude fastly, cold_starts = 1)
providers_cold = ["google", "awsX86", "awsARM", "oracleX86", "oracleARM", "deno", "cloudflare", "flyio"]
column = "http_req_waiting"

def get_provider_df_cold(cnx, provider: str) -> pd.DataFrame:
    """Load and lightly clean provider data for coldstart (cold_starts=1)."""
    query = """
    SELECT http_req_waiting, language, memory
    FROM data
    WHERE provider = ?
      AND experiment = 'coldstart'
      AND cold_starts = 1
    """
    params = [provider]

    df = pd.read_sql_query(query, cnx, params=params)

    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found for provider '{provider}'.")

    df[column] = pd.to_numeric(df[column], errors="coerce")
    df = df.dropna(subset=[column, "memory", "language"])

    return df

def get_provider_df_warm_for_ratio(cnx, provider: str) -> pd.DataFrame:
    """Load warmstart data for coldstart ratio (cold_starts=0)."""
    query = """
    SELECT http_req_waiting, language, memory
    FROM data
    WHERE provider = ?
      AND experiment = 'warmstart'
      AND cold_starts = 0
    """
    params = [provider]

    df = pd.read_sql_query(query, cnx, params=params)

    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found for provider '{provider}'.")

    df[column] = pd.to_numeric(df[column], errors="coerce")
    df = df.dropna(subset=[column, "memory", "language"])

    return df

all_results_cold = []

for p in providers_cold:
    print("\n" + "=" * 90)
    print(f"PROVIDER (coldstart): {p}")
    print("=" * 90)

    df_p = get_provider_df_cold(cnx, p)

    if df_p.empty:
        print("No data after filtering/cleaning.")
        continue

    table = provider_summary_table(df_p, p)

    df_p_warm = get_provider_df_warm_for_ratio(cnx, p)
    if not df_p_warm.empty:
        table_warm = provider_summary_table(df_p_warm, p)
        warm_medians = table_warm[["provider", "level", "memory", "language", "median"]].rename(
            columns={"median": "median_warm"}
        )
        table = table.merge(warm_medians, on=["provider", "level", "memory", "language"], how="left")
        table["coldstart_factor"] = table["median"] / table["median_warm"]
        table = table.drop(columns=["median_warm"], errors="ignore")
    else:
        table["coldstart_factor"] = float("nan")

    display_cols = ["level", "memory", "language", "count", "median", "mean", "std", "p25", "p75", "p95", "p99", "iqr", "tail_ratio", "coldstart_factor"]
    print(table[display_cols].to_string(index=False))

    all_results_cold.append(table)

combined_cold = pd.concat(all_results_cold, ignore_index=True) if all_results_cold else pd.DataFrame()

keys = ["provider", "level", "memory", "language"]
combined_warm = combined.copy()
cold_medians = (
    combined_cold[keys + ["median"]]
    .rename(columns={"median": "median_cold"})
)
combined_warm = combined_warm.merge(cold_medians, on=keys, how="left")
combined_warm["coldstart_factor"] = combined_warm["median_cold"] / combined_warm["median"]
combined_warm = combined_warm.drop(columns=["median_cold"], errors="ignore")

combined_warm


PROVIDER (coldstart): google
          level memory language  count     median       mean        std        p25         p75         p95         p99        iqr  tail_ratio  coldstart_factor
    overall_raw    all      all   2426 533.728680 567.116370 326.844199 329.488946  701.702930 1093.678606 1706.546155 372.213984    3.009164         12.550782
   overall_fair    all      all   2426 511.328563 612.316650 234.170775 450.568148  577.740391  832.262454 1149.299270 134.827296    2.624048         12.043705
         memory 1024MB      all    805 396.942915 517.037983 388.019117 320.043857  573.362729 1190.121221 1905.339492 253.318872    3.685105          9.343954
         memory  128MB      all    610 687.320752 674.187115 357.428677 372.024198  825.977778 1262.377022 1991.232327 453.953579    2.953531         16.082112
         memory  512MB      all   1011 583.245658 542.388327 223.769220 320.611798  682.721591  846.715160 1128.598704 362.109793    2.080795         13.760459
       lan

,provider,level,memory,language,count,mean,median,std,p25,p75,p95,p99,iqr,tail_ratio,coldstart_factor
0,google,overall_raw,all,all,1919360,45.904602,42.525534,30.721298,41.266987,47.500697,55.075430,110.091100,6.233711,2.398258,12.550782
1,google,overall_fair,all,all,1919360,45.904418,42.456086,26.555182,41.306052,47.306006,52.307725,66.691828,5.791124,1.505308,12.043705
2,google,memory,1024MB,all,639603,44.644648,42.481256,37.411556,41.291281,47.097015,51.918538,66.700075,5.805734,1.494022,9.343954
3,google,memory,128MB,all,639918,48.549527,42.738216,22.307390,41.355141,48.438763,83.807667,133.694587,7.083622,2.753777,16.082112
4,google,memory,512MB,all,639839,44.518838,42.385624,30.395275,41.158554,46.763570,51.930892,66.284412,5.605016,1.488907,13.760459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,fastly,memory,na,all,640156,11.246481,8.941088,92.782407,7.365020,10.862982,17.561679,45.142443,3.497962,4.013917,NaN
91,fastly,language,all,go,320083,12.026494,10.124105,22.878713,8.892096,11.895150,18.856653,44.809176,3.003054,3.725872,NaN
92,fastly,language,all,js,320073,10.466443,7.612048,129.200531,6.590464,9.010774,15.575883,45.517961,2.420310,4.348943,NaN
93,fastly,memory+language,na,go,320083,12.026494,10.124105,22.878713,8.892096,11.895150,18.856653,44.809176,3.003054,3.725872,NaN
